In [3]:
# Calculating feature importance of the dataset
#  Only look at the dataset with valid genres for the time being
import pandas as pd 
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import DictVectorizer


In [5]:
# normalise the dataset and create a new csv
data = pd.read_csv('data/songs_with_super_genres.csv')

# find the minimum value per column, which may be negative, if its negative add that to to all numbers such that the bottom becomes zero, u want the bottom value to be zero
# find the new maximum value and divide every value the max and that will give u a scale between 0 and 1. 

# function to find the minimum value of the specified dataframe column 
def find_minimum_value(column):
    # find the minimum value
    return column.min()

# apply the function to the whole dataset

for index, row in data.iterrows():
    for col in data.select_dtypes(include=[np.number]).columns:
        # find the minimum value
        min_val = find_minimum_value(data[col])
        # shift the values to make the minimum zero
        shifted = data[col] - min_val
        # find the maximum value
        max_val = shifted.max()
        # divide by the maximum value if it's not zero
        if max_val != 0:
            data[col] = shifted / max_val

data.to_csv('data/normalised_data.csv', index=False)


In [6]:
data = pd.read_csv('data/normalised_data.csv')

valid_genre_data_mask = pd.isna(data['super_genre']) == False
valid_genre_data = data[valid_genre_data_mask]
valid_genre_data

# Split the data into training set 80% and test set 20%
df_train_full, df_test = train_test_split(valid_genre_data, test_size=0.2, random_state=1)
# Further split the training set into training set 75% and validation set 25%
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=1)

# train_test_split() also shuffles the data randomly before splitting
# so resetting the indexes as follows:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

#  extract the target labels (the genre column) as Numpy arrays
y_train = df_train['super_genre'].values
y_val = df_val['super_genre'].values
y_test = df_test['super_genre'].values

#  drop the target label from the data
#  for now dont include one-hot enocodings for nationality and detected language, maybe come back and include one hot encodings?
del df_train['super_genre']
del df_val['super_genre']
del df_test['super_genre']

cols = valid_genre_data.select_dtypes(include=np.number).columns.tolist()
print(cols)
#  ['Rank', 'Danceability', 'Energy', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Valence', 
# 'Points (Total)', 'num_languages', 'track_popularity', 'artist_followers', 'artist_popularity', 'Loudness']

['Rank', 'Danceability', 'Energy', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Valence', 'Points (Total)', 'num_languages', 'track_popularity', 'artist_followers', 'artist_popularity', 'Loudness']


In [7]:
# change the unique genres into numerical labels by numbering the super genres from 0 to n-1, n being the total number of unique super genres
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

#  gets the numeric feature columns and converts the resulting dataframe into a list of dictionaries, e.g., 'danceability': 0.5, 'energy': 0.8, ...
train_dicts = df_train[cols].to_dict(orient='records')

#  DictVectorizer will convert the list of dictionaries into a numeric vector similar to one-hot encoding for categorical variables, a dense Numpy array
#  fit_transform both learns the structure of waht features exists and converts the training data
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

# we do a similar thing for the validation data
val_dict = df_val[cols].to_dict(orient='records')
X_val = dv.transform(val_dict)  # only transform, don't fit again


In [8]:
# Feature importance using Random Forest Classifier
rf = RandomForestClassifier(random_state=1)
rf.fit(X_train, y_train)
importances = rf.feature_importances_
print(importances)

df_importances = pd.DataFrame()
df_importances['feature'] = dv.feature_names_
df_importances['importance'] = importances
df_importances.sort_values(by='importance', ascending=False, inplace=True)
print(df_importances)

[0.07855884 0.08262364 0.07887034 0.01605744 0.08523837 0.04812991
 0.04801992 0.10427404 0.07966028 0.17487903 0.13474247 0.00898191
 0.05996381]
              feature  importance
9    artist_followers    0.174879
10  artist_popularity    0.134742
7         Speechiness    0.104274
4            Loudness    0.085238
1        Danceability    0.082624
8             Valence    0.079660
2              Energy    0.078870
0        Acousticness    0.078559
12   track_popularity    0.059964
5      Points (Total)    0.048130
6                Rank    0.048020
3    Instrumentalness    0.016057
11      num_languages    0.008982


In [9]:
#print the size of train, val and test datasets
print(len(df_train))
print(len(df_val))
print(len(df_test))

3152
1051
1051


In [10]:
#Implementing a Logistic Regression Model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

#baseline lr model
lr_model = LogisticRegression(multi_class='ovr', solver='lbfgs', random_state=1)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict_proba(X_val)
print("baseline score")
print(roc_auc_score(y_val, y_pred, multi_class='ovr'))

baseline score
0.8572213868421467


/Users/victoria/Documents/University/AML/AML_Spotify/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


In [11]:
#balanced class lr model with max iterations increased
lr_model = LogisticRegression(multi_class='ovr', solver='lbfgs', random_state=1, class_weight='balanced', max_iter=550)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict_proba(X_val)
print("balanced class score")
print(roc_auc_score(y_val, y_pred, multi_class='ovr'))
#Making the class weights balanced improved the score

balanced class score
0.853815452587137


/Users/victoria/Documents/University/AML/AML_Spotify/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


In [12]:
# List of different max_iter values to try
max_iters = [600, 700, 800, 900, 1000, 1500, 2000]

# Store results
scores = []

for iters in max_iters:
    # Create and fit the logistic regression model
    lr_model = LogisticRegression(
        multi_class='ovr',
        solver='lbfgs',
        random_state=1,
        class_weight='balanced',
        max_iter=iters
    )
    lr_model.fit(X_train, y_train)
    
    # Predict probabilities on the validation set
    y_pred = lr_model.predict_proba(X_val)
    
    # Calculate ROC AUC score
    auc = roc_auc_score(y_val, y_pred, multi_class='ovr')
    
    # Append results
    scores.append((iters, auc))

# Create a DataFrame for easy inspection
df_scores = pd.DataFrame(scores, columns=['max_iter', 'auc'])
df_scores.round(3)
#Increasing the max iterations doesn't change anything

/Users/victoria/Documents/University/AML/AML_Spotify/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/victoria/Documents/University/AML/AML_Spotify/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/victoria/Documents/University/AML/AML_Spotify/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this 

,max_iter,auc
0,600,0.854
1,700,0.854
2,800,0.854
3,900,0.854
4,1000,0.854
5,1500,0.854
6,2000,0.854


In [14]:
#Implementing a decision tree classifier
from sklearn.tree import DecisionTreeClassifier

scores = []

for depth in [5, 7, 10, 12, 15, 17]:
    for s in [1, 5, 10, 15, 20, 500, 100, 200]:
        dt = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=s)
        dt.fit(X_train, y_train)
        
        y_pred = dt.predict_proba(X_val)
        auc = roc_auc_score(y_val, y_pred, multi_class='ovr')
        
        scores.append((depth, s, auc))

columns = ['max_depth', 'min_samples_leaf', 'auc']
df_scores = pd.DataFrame(scores, columns=columns)

df_scores_pivot = df_scores.pivot(index='min_samples_leaf', columns=['max_depth'], values=['auc'])
df_scores_pivot.round(3)

#highest score is with max_depth=7 and min_samples_leaf=20 which is 0.859

auc                                   
max_depth            5      7      10     12     15     17
min_samples_leaf                                          
1                 0.819  0.820  0.807  0.783  0.752  0.746
5                 0.803  0.828  0.823  0.811  0.802  0.805
10                0.803  0.823  0.822  0.824  0.824  0.822
15                0.818  0.850  0.858  0.856  0.855  0.855
20                0.817  0.859  0.850  0.850  0.846  0.846
100               0.829  0.847  0.846  0.846  0.846  0.846
200               0.800  0.800  0.800  0.800  0.800  0.800
500               0.773  0.773  0.773  0.773  0.773  0.773